<a id="inicio"></a>
    # AED 07 · De los hallazgos a una decisión defendible

    **Núcleo U3 · S10–S12 · RA3 / CE3.1–3.4** · Colección docente · Edición 2026-10-08.

    Ejecute todas las celdas en orden con el entorno documentado en `../reproducibilidad/README.md`.
    Los controles necesitan kernel activo; las salidas guardadas permiten lectura. Los datos son
    casos docentes locales; las simulaciones y parámetros económicos se identifican como supuestos.
    Consulte la [guía maestra](../guia_maestra_aed.html), el
    [manual científico](../material_propio/AED_manual_cientifico.pdf) y la
    [matriz curricular](../COBERTURA.md). Los ejercicios son formativos.

### Antes de ejecutar

Núcleo · 90–120 min orientativos.

**Objetivo:** Defender una alternativa y diseñar el piloto que evaluará su efecto.

**Preparación:** KPIs, restricciones y análisis de incertidumbre de los módulos anteriores.

**Ejemplo de referencia:** Traducir una reducción supuesta de incidentes a beneficio incremental y punto de equilibrio.

**Práctica:** Cambiar costo y efecto esperado; formular una métrica primaria y una de protección.

**Criterio de logro:** Entregar memo con alternativas, factibilidad, incertidumbre, piloto, responsable y criterio de revisión.

[Guía y secuencia](../guia_maestra_aed.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/AED_manual_cientifico.html#sec-comunicacion)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
RAIZ=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'_transversal/datos').is_dir())
sys.path.insert(0,str(RAIZ/'04_Analitica_Estrategica_Datos/reproducibilidad'))
from aed_recursos import (casapeumo, indicadores, boldonet, plan_produccion,
                         decision, solar, resumen_solar, pregunta, DATOS, SEMILLA)
plt.rcParams.update({'figure.figsize':(8,3.5),'axes.spines.top':False,'axes.spines.right':False})

<a id="argumento"></a>
## 1. Estructura de un argumento ejecutivo

Decisión solicitada → evidencia → alternativas → incertidumbre → acción y seguimiento.
Una narrativa persuasiva permite cuestionar los supuestos y reconocer qué haría cambiar la
recomendación. No se trata de ocultar límites para conseguir aprobación. El directorio necesita
impacto, recursos y exposición; operaciones necesita regla de trabajo, capacidad y excepciones.
Los números deben coincidir en ambas versiones.

Usaremos CasaPeumo para seleccionar un problema de servicio, explicitar el salto entre hallazgo
descriptivo e intervención causal y preparar un piloto medible. Los datos no contienen resultados
de un experimento: cualquier mejora atribuida a la intervención es un supuesto.

In [2]:
ventas,_,auditoria=casapeumo()
filas=[]
for canal,f in ventas.groupby('Canal'):
    filas.append({'canal':canal,**indicadores(f)})
evidencia=pd.DataFrame(filas)
display(evidencia)
foco=evidencia.sort_values('reclamos_por_1000',ascending=False).iloc[0]
print('Foco descriptivo para investigar:',foco.canal)
assert sum(evidencia.ventas)==len(ventas)

,canal,ventas,ingreso,margen,margen_pct,ticket,reclamos_por_1000,entrega_media
0,Marketplace,138,23555.9741,7249.6341,0.307762,170.695464,43.478261,1.384058
1,Sitio web,275,40978.4763,12242.1163,0.298745,149.012641,54.545455,1.563636
2,Tienda física,262,36971.1607,10849.6507,0.293463,141.111300,68.702290,0.000000


Foco descriptivo para investigar: Tienda física


<a id="alternativas"></a>
## 2. Comparar intervención con no actuar

La comparación debe usar costos incrementales y el mismo horizonte. Ejemplo ficticio: 5000
pedidos, tasa base de reclamos 8%, costo por reclamo 20 UM; piloto cuesta 1000 UM y se supone
que reduce 20% el riesgo relativo. La reducción absoluta es 0.08 × 0.20 = 0.016, no 20 puntos
porcentuales. Beneficio neto esperado = N × tasa base × reducción relativa × costo evitado − costo.
La tasa y el efecto de este ejemplo son supuestos, separados del tablero observado de CasaPeumo.

In [3]:
@widgets.interact(reduccion=widgets.FloatSlider(value=.2,min=0,max=.5,step=.02),costo=widgets.IntSlider(value=1000,min=0,max=4000,step=100))
def piloto(reduccion=.2,costo=1000):
    evitados=5000*.08*reduccion;neto=evitados*20-costo
    display(pd.Series({'reclamos_evitados_esperados':evitados,'beneficio_neto_supuesto':neto,
                      'reduccion_absoluta_pp':100*.08*reduccion}))
    assert evitados>=0
umbral=1000/(5000*.08*20)
assert np.isclose(umbral,.125)
print('Reducción relativa de equilibrio:',umbral)

interactive(children=(FloatSlider(value=0.2, description='reduccion', max=0.5, step=0.02), IntSlider(value=100…

Reducción relativa de equilibrio: 0.125


<a id="experimento"></a>
## 3. Diseñar evaluación antes de implementar

Asigne aleatoriamente unidades elegibles a control/tratamiento cuando sea viable; si la acción
opera por sucursal, aleatorice por sucursal y modele dependencia. Defina KPI principal,
guardarraíles, ventana y análisis antes de mirar resultados. El cálculo simple de tamaño siguiente
supone grupos independientes de igual tamaño, aproximación normal, alfa 5% bilateral y potencia
80%. No incorpora correlación intraclúster, pérdidas ni pruebas múltiples: es un punto inicial
para planificar, no una garantía. Una muestra insuficiente no demuestra ausencia de efecto.

In [4]:
from scipy.stats import norm
p0=.08;p1=.064;pm=(p0+p1)/2
n_grupo=((norm.ppf(.975)*np.sqrt(2*pm*(1-pm))+norm.ppf(.8)*np.sqrt(p0*(1-p0)+p1*(1-p1)))**2/(p0-p1)**2)
print('Aproximación de observaciones por grupo:',int(np.ceil(n_grupo)))
plan=pd.DataFrame([
 ['Principal','Ventas con reclamo / ventas','90 días','Operaciones'],
 ['Guardarraíl','Margen bruto / ingreso','Mismo período','Finanzas'],
 ['Calidad','Claves completas / registros','Semanal','Datos'],
 ['Implementación','Pedidos que recibieron protocolo / asignados','Semanal','Líder del piloto']
],columns=['rol','definicion','ventana','responsable'])
display(plan)

Aproximación de observaciones por grupo: 4096


,rol,definicion,ventana,responsable
0,Principal,Ventas con reclamo / ventas,90 días,Operaciones
1,Guardarraíl,Margen bruto / ingreso,Mismo período,Finanzas
2,Calidad,Claves completas / registros,Semanal,Datos
3,Implementación,Pedidos que recibieron protocolo / asignados,Semanal,Líder del piloto


<a id="narrativa"></a>
## 4. Un gráfico, una comparación y una limitación

Las barras parten de cero, muestran unidad y denominador; los porcentajes incluyen tamaño de
grupo. No use color como único significado. Para explicar a operaciones agregue acción y
excepción; para dirección agregue costo y condición de continuación. Mantenga un apéndice técnico
con reglas de calidad, fórmulas, versiones y alternativas descartadas.

In [5]:
@widgets.interact(audiencia=['Directorio','Operaciones'])
def comunicar(audiencia='Directorio'):
    fig,ax=plt.subplots();ax.bar(evidencia.canal,evidencia.reclamos_por_1000,color='#176b78')
    for i,row in evidencia.iterrows():ax.text(i,row.reclamos_por_1000+1,f'n={int(row.ventas)}',ha='center')
    ax.set(ylabel='Ventas con reclamo por 1000 ventas',title='CasaPeumo: descripción del semestre');plt.show();plt.close(fig)
    texto=('Solicitar un piloto acotado; efecto y retorno aún no medidos.' if audiencia=='Directorio'
           else 'Investigar causas de reclamo y definir protocolo, responsables y excepciones antes del piloto.')
    display(Markdown('**'+audiencia+':** '+texto+' La diferencia entre canales no identifica causalidad.'))

interactive(children=(Dropdown(description='audiencia', options=('Directorio', 'Operaciones'), value='Director…

<a id="entrega"></a>
## 5. Caso integrador y defensa

Entregue un memo de una página y cinco diapositivas: decisión, evidencia, alternativas, piloto,
seguimiento. Adjunte notebook y diccionario para reproducir. Una defensa sólida contesta:
¿por qué ese denominador?, ¿qué dato falta?, ¿qué alternativa fue rechazada y por qué?, ¿qué
resultado detendría el piloto?, ¿quién implementa?, ¿cómo se protegerán los registros individuales?

**Solución orientadora:** conservar operación actual como base; comparar piloto y despliegue
completo con costos/efectos explícitos; elegir un piloto si falta evidencia causal; asignar dueño y
ventana; no escalar hasta confirmar efecto compatible con rentabilidad y servicio. Para el embudo
digital, diferencie conversión de sesiones y conversión de inicios de pago: ambos son válidos,
pero responden preguntas distintas. No sumar tasas de segmentos.

In [6]:
digital=pd.read_excel(DATOS/'embudo_digital/originales/embudo_digital.xlsx',sheet_name='Datos originales')
assert np.allclose(digital.Inicios_pago,digital.Compras+digital.Abandonos_pago)
digital['conversion_sesion']=digital.Compras/digital.Sesiones_actual
digital['conversion_checkout']=digital.Compras/digital.Inicios_pago
display(digital[['Segmento','Sesiones_actual','Inicios_pago','Compras','conversion_sesion','conversion_checkout']])
print('Conversión global por sesión:',digital.Compras.sum()/digital.Sesiones_actual.sum())

,Segmento,Sesiones_actual,Inicios_pago,Compras,conversion_sesion,conversion_checkout
0,Web escritorio,14500,1232,887,0.061172,0.719968
1,Web móvil Android,12800,794,437,0.034141,0.550378
2,Web móvil iOS,11900,845,532,0.044706,0.629586
3,App Android,9100,892,696,0.076484,0.780269
4,App iOS,8600,894,724,0.084186,0.809843
5,Tablet,4100,303,200,0.048780,0.660066


Conversión global por sesión: 0.056983606557377046


<a id="autoevaluacion"></a>
## Autoevaluación
Seleccione y compruebe su respuesta.

In [7]:
auto_07=pregunta('¿Qué debe acompañar una recomendación ejecutiva?', ['Solo la alternativa preferida', 'Evidencia, alternativas, incertidumbre, responsable y seguimiento', 'La mayor cantidad posible de gráficos'], 1, 'Una decisión defendible muestra qué la sustenta, sus límites y cómo se comprobará su efecto.')